In [1]:
pip install folium branca


Note: you may need to restart the kernel to use updated packages.


In [2]:
# =============================================================
#  PUNE NO2 — INTERACTIVE HIGH RESOLUTION HEATMAP
#
#  Creates a single HTML file you can open in any browser.
#  Features:
#    - Real map tiles (Street / Satellite / Terrain)
#    - Colour-coded NO2 circles at every predicted pixel
#    - Click any pixel → popup shows NO2 value + AQI status
#    - Hover to see value instantly
#    - Layer toggle: Annual mean / Winter / Summer / Monsoon
#    - Date slider to see daily maps
#    - Ground station markers with measured values
#    - WHO and NAAQS limit toggle lines on colour bar
#    - Download selected data as CSV
#
#  Install:
#    pip install folium pandas numpy branca
#
#  Run:
#    python3 create_interactive_map.py
#
#  Output:
#    map_outputs/pune_no2_interactive.html   ← open in browser
# =============================================================

import pandas as pd
import numpy as np
import folium
from folium.plugins import HeatMap, MousePosition, MeasureControl, MiniMap
import branca.colormap as cm
import json
import os
import warnings
warnings.filterwarnings("ignore")

os.makedirs("map_outputs", exist_ok=True)

# ── FILE PATHS — edit if yours are different ───────────────────
# Priority order: use 500m predictions if available, else fall back
PRED_PATHS = [
    "data/output/all_predictions_500m.csv",   # 500m fine resolution
    "map_outputs/all_predictions.csv",         # 5km fallback
]

GND_PATH = "data/Preprocessed/ground_clean.csv"

OUTPUT_HTML = "map_outputs/pune_no2_interactive.html"

# AQI thresholds for NO2 (µg/m³) — CPCB India categories
AQI_LEVELS = [
    (0,   40,  "#00E400", "Good",            "Safe — no health concern"),
    (40,  80,  "#FFFF00", "Satisfactory",    "Minor concern for sensitive groups"),
    (80,  120, "#FF7E00", "Moderate",        "Exceeds NAAQS — moderate risk"),
    (120, 180, "#FF0000", "Poor",            "Health effects for all"),
    (180, 250, "#8F3F97", "Very Poor",       "Serious health effects"),
    (250, 999, "#7E0023", "Severe",          "Emergency — severe health effects"),
]

WHO_LIMIT   = 25.0    # WHO 2021 annual guideline
NAAQS_LIMIT = 80.0    # India NAAQS annual standard

STATIONS = {
    "Gavalinagar":   (18.63673, 73.82487),
    "Katraj Dairy":  (18.45445, 73.85416),
    "Park Wakad":    (18.59700, 73.76200),
    "SPPU":          (18.52900, 73.85100),
    "Thergaon":      (18.61100, 73.78400),
    "Bhumkar Nagar": (18.59800, 73.77300),
    "Hadapsar":      (18.50300, 73.93900),
    "Panchawati":    (18.53600, 73.82600),
    "Savta Mali":    (18.62800, 73.80600),
    "Nigdi":         (18.65400, 73.78800),
}

SEASON_NAMES = {0:"Winter", 1:"Summer", 2:"Monsoon", 3:"Post-monsoon"}


# =============================================================
#  LOAD PREDICTIONS
# =============================================================

def load_predictions():
    for path in PRED_PATHS:
        if os.path.exists(path):
            print(f"Loading predictions from: {path}")
            df = pd.read_csv(path)
            print(f"  Rows: {len(df):,}  |  Columns: {df.columns.tolist()}")

            # Ensure required columns
            if "NO2_pred_ugm3" not in df.columns:
                # If using ml_ready.csv directly (training data with ground truth)
                if "NO2_ground_ugm3" in df.columns:
                    df["NO2_pred_ugm3"] = df["NO2_ground_ugm3"]
                    print("  Using NO2_ground_ugm3 as prediction column")

            # Ensure season column
            if "season" not in df.columns and "month" in df.columns:
                df["season"] = df["month"].map(
                    {12:0,1:0,2:0,3:1,4:1,5:1,6:2,7:2,8:2,9:2,10:3,11:3}
                )

            # Rename sat_lat/sat_lon if needed
            if "sat_lat" in df.columns:
                df = df.rename(columns={"sat_lat":"latitude","sat_lon":"longitude"})

            return df

    print("⚠️  No prediction file found. Checking for training data...")
    # Last fallback — use ml_ready.csv
    for path in ["data/Preprocessed/ml_ready.csv", "ml_ready.csv",
                 "data/output/training_data_500m.csv"]:
        if os.path.exists(path):
            df = pd.read_csv(path)
            if "NO2_ground_ugm3" in df.columns:
                df["NO2_pred_ugm3"] = df["NO2_ground_ugm3"]
                print(f"  Using {path} with ground truth as prediction")
                return df

    raise FileNotFoundError(
        "No prediction CSV found.\n"
        "Expected one of:\n"
        "  data/output/all_predictions_500m.csv\n"
        "  map_outputs/all_predictions.csv\n"
        "Run step2_train_and_predict.py first."
    )


# =============================================================
#  AQI HELPER FUNCTIONS
# =============================================================

def get_aqi_color(no2_val):
    for lo, hi, color, label, desc in AQI_LEVELS:
        if lo <= no2_val < hi:
            return color, label, desc
    return "#7E0023", "Severe", "Emergency conditions"


def get_who_status(no2_val):
    if no2_val <= WHO_LIMIT:
        return f"<span style='color:#00aa00'>✓ Within WHO guideline (≤{WHO_LIMIT})</span>"
    elif no2_val <= NAAQS_LIMIT:
        return f"<span style='color:#ff8800'>⚠ Exceeds WHO ({WHO_LIMIT}), within NAAQS ({NAAQS_LIMIT})</span>"
    else:
        return f"<span style='color:#cc0000'>✗ Exceeds NAAQS limit ({NAAQS_LIMIT})</span>"


def make_popup_html(row, station_name=None):
    no2  = row["NO2_pred_ugm3"]
    color, label, desc = get_aqi_color(no2)
    who_status         = get_who_status(no2)

    date_str = f"<b>Date:</b> {row.get('date','Annual mean')}<br>" \
               if "date" in row and pd.notna(row.get("date")) else ""

    station_str = f"<b>Station:</b> {station_name}<br>" if station_name else ""

    return f"""
    <div style='font-family:Arial,sans-serif;font-size:13px;
                min-width:220px;max-width:260px'>
      <div style='background:{color};color:{"#000" if color in ["#00E400","#FFFF00"] else "#fff"};
                  padding:6px 10px;border-radius:4px 4px 0 0;
                  font-weight:bold;font-size:14px'>
        NO₂: {no2:.1f} µg/m³ — {label}
      </div>
      <div style='padding:8px 10px;border:1px solid #ddd;
                  border-top:none;border-radius:0 0 4px 4px;
                  background:#fafafa'>
        {date_str}
        {station_str}
        <b>AQI category:</b> {label}<br>
        <span style='color:#666;font-size:11px'>{desc}</span><br><br>
        {who_status}<br><br>
        <b>Location:</b> {row['latitude']:.5f}°N,
                          {row['longitude']:.5f}°E<br>
        <span style='color:#888;font-size:10px'>
          WHO annual guideline: {WHO_LIMIT} µg/m³ |
          India NAAQS: {NAAQS_LIMIT} µg/m³
        </span>
      </div>
    </div>
    """


# =============================================================
#  BUILD THE MAP
# =============================================================

def build_interactive_map(df):
    print("\nBuilding interactive map...")

    # Annual mean per pixel for the default view
    annual = df.groupby(["latitude","longitude"],
                         as_index=False)["NO2_pred_ugm3"].mean()
    annual["NO2_pred_ugm3"] = annual["NO2_pred_ugm3"].round(2)

    print(f"  Unique pixels: {len(annual):,}")
    print(f"  NO2 range: {annual['NO2_pred_ugm3'].min():.1f} – "
          f"{annual['NO2_pred_ugm3'].max():.1f} µg/m³")

    # ── BASE MAP ────────────────────────────────────────────────
    m = folium.Map(
        location=[18.52, 73.86],
        zoom_start=12,
        tiles=None,
        control_scale=True
    )

    # ── TILE LAYERS ─────────────────────────────────────────────
    folium.TileLayer(
        tiles="OpenStreetMap",
        name="Street map",
        control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=s&x={x}&y={y}&z={z}",
        attr="Google Satellite",
        name="Satellite view",
        control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="https://mt1.google.com/vt/lyrs=m&x={x}&y={y}&z={z}",
        attr="Google Maps",
        name="Google Maps",
        control=True
    ).add_to(m)

    folium.TileLayer(
        tiles="CartoDB positron",
        name="Light (CartoDB)",
        control=True
    ).add_to(m)

    # ── COLOUR SCALE ─────────────────────────────────────────────
    vmin = 0
    vmax = min(float(annual["NO2_pred_ugm3"].quantile(0.97)), 200)

    colormap = cm.LinearColormap(
        colors=["#00E400","#FFFF00","#FF7E00","#FF0000","#8F3F97","#7E0023"],
        vmin=vmin, vmax=vmax,
        caption="Predicted surface NO₂ (µg/m³) — CPCB AQI scale"
    )
    colormap.add_to(m)

    # ── LAYER 1: ANNUAL MEAN NO2 ────────────────────────────────
    print("  Adding annual mean layer...")
    annual_layer = folium.FeatureGroup(name="NO₂ — Annual mean", show=True)

    # Determine marker size based on pixel count
    # More pixels = smaller markers to avoid overlap
    n_pixels = len(annual)
    if n_pixels > 10000:
        radius, weight = 4,  0.2
    elif n_pixels > 2000:
        radius, weight = 6,  0.3
    else:
        radius, weight = 8,  0.5

    for _, row in annual.iterrows():
        no2   = row["NO2_pred_ugm3"]
        color = colormap(no2)

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.80,
            weight=weight,
            popup=folium.Popup(make_popup_html(row), max_width=280),
            tooltip=f"NO₂: {no2:.1f} µg/m³"
        ).add_to(annual_layer)

    annual_layer.add_to(m)

    # ── LAYER 2: HEATMAP OVERLAY ────────────────────────────────
    # Provides smooth gradient view — good for overview
    print("  Adding heatmap overlay layer...")
    heat_data = [[row["latitude"], row["longitude"],
                  float(row["NO2_pred_ugm3"])]
                 for _, row in annual.iterrows()]

    HeatMap(
        heat_data,
        name="NO₂ — Heatmap (smooth)",
        show=False,
        min_opacity=0.3,
        max_zoom=16,
        radius=15,
        blur=12,
        gradient={
            "0.0": "#00E400",
            "0.3": "#FFFF00",
            "0.6": "#FF7E00",
            "0.8": "#FF0000",
            "1.0": "#7E0023"
        }
    ).add_to(m)

    # ── LAYER 3–6: SEASONAL LAYERS ──────────────────────────────
    if "season" in df.columns:
        print("  Adding seasonal layers...")
        for s_code, s_name in SEASON_NAMES.items():
            sub = df[df["season"] == s_code]
            if len(sub) == 0:
                continue

            season_mean = sub.groupby(["latitude","longitude"],
                                       as_index=False)["NO2_pred_ugm3"].mean()
            season_mean["NO2_pred_ugm3"] = season_mean["NO2_pred_ugm3"].round(2)
            season_mean["date"]          = s_name

            s_layer = folium.FeatureGroup(
                name=f"NO₂ — {s_name} mean",
                show=False
            )

            for _, row in season_mean.iterrows():
                no2   = row["NO2_pred_ugm3"]
                color = colormap(no2)
                folium.CircleMarker(
                    location=[row["latitude"], row["longitude"]],
                    radius=radius,
                    color=color,
                    fill=True,
                    fill_color=color,
                    fill_opacity=0.80,
                    weight=weight,
                    popup=folium.Popup(make_popup_html(row), max_width=280),
                    tooltip=f"{s_name}: {no2:.1f} µg/m³"
                ).add_to(s_layer)

            s_layer.add_to(m)

    # ── LAYER 7: EXCEEDANCE LAYER ────────────────────────────────
    print("  Adding NAAQS exceedance layer...")
    exc_data = df.groupby(["latitude","longitude"],
                           as_index=False).apply(
        lambda g: pd.Series({
            "pct_naaqs": round((g["NO2_pred_ugm3"] > NAAQS_LIMIT).mean()*100, 1),
            "pct_who":   round((g["NO2_pred_ugm3"] > WHO_LIMIT).mean()*100,   1),
        })
    ).reset_index(drop=True)

    exc_layer = folium.FeatureGroup(
        name="NAAQS exceedance (% days)", show=False
    )

    exc_cmap = cm.LinearColormap(
        colors=["#ffffcc","#fed976","#fd8d3c","#e31a1c","#800026"],
        vmin=0, vmax=100,
        caption="% days exceeding NAAQS (80 µg/m³)"
    )

    for _, row in exc_data.iterrows():
        pct   = row["pct_naaqs"]
        color = exc_cmap(pct)
        popup_html = f"""
        <div style='font-family:Arial;font-size:13px;min-width:200px'>
          <b>NAAQS Exceedance</b><br>
          % days above 80 µg/m³: <b style='color:{"#cc0000" if pct>50 else "#ff8800"}'>{pct:.1f}%</b><br>
          % days above WHO 25 µg/m³: <b>{row['pct_who']:.1f}%</b><br>
          Location: {row['latitude']:.5f}°N, {row['longitude']:.5f}°E
        </div>
        """
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.80,
            weight=weight,
            popup=folium.Popup(popup_html, max_width=240),
            tooltip=f"Exceeds NAAQS: {pct:.1f}% of days"
        ).add_to(exc_layer)

    exc_layer.add_to(m)

    # ── LAYER 8: GROUND STATION MARKERS ─────────────────────────
    print("  Adding ground station markers...")
    station_layer = folium.FeatureGroup(name="Ground stations", show=True)

    # Load measured values if available
    gnd_means = {}
    if os.path.exists(GND_PATH):
        gnd = pd.read_csv(GND_PATH)
        gnd_means = gnd.groupby("location_name")["NO2_ground_ugm3"].mean().round(1).to_dict()

    for name, (lat, lon) in STATIONS.items():
        measured = gnd_means.get(name, None)
        meas_str = f"<br><b>Measured mean:</b> {measured:.1f} µg/m³" if measured else ""

        popup_html = f"""
        <div style='font-family:Arial;font-size:13px;min-width:200px'>
          <b style='font-size:14px'>📍 {name}</b><br>
          <span style='color:#666'>CPCB/MPCB Monitoring Station</span>
          {meas_str}<br>
          Lat: {lat:.5f}°N  |  Lon: {lon:.5f}°E
        </div>
        """

        folium.Marker(
            location=[lat, lon],
            icon=folium.Icon(color="darkblue", icon="info-sign"),
            popup=folium.Popup(popup_html, max_width=240),
            tooltip=f"Station: {name}"
        ).add_to(station_layer)

    station_layer.add_to(m)

    # ── LAYER 9: WHO GUIDELINE CIRCLES (25 µg/m³ zones) ─────────
    # Show areas consistently within/exceeding WHO limit
    who_layer = folium.FeatureGroup(name="WHO compliance zones", show=False)
    who_data  = df.groupby(["latitude","longitude"],
                            as_index=False)["NO2_pred_ugm3"].mean()

    for _, row in who_data.iterrows():
        no2   = row["NO2_pred_ugm3"]
        color = "#00aa00" if no2 <= WHO_LIMIT else \
                "#ff8800" if no2 <= NAAQS_LIMIT else "#cc0000"
        label = "WHO compliant" if no2 <= WHO_LIMIT else \
                "Between WHO–NAAQS" if no2 <= NAAQS_LIMIT else "Exceeds NAAQS"

        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=radius,
            color=color,
            fill=True,
            fill_color=color,
            fill_opacity=0.70,
            weight=weight,
            tooltip=f"{label}: {no2:.1f} µg/m³"
        ).add_to(who_layer)

    who_layer.add_to(m)

    # ── PLUGINS ─────────────────────────────────────────────────
    # Mouse position — shows lat/lon as you hover
    MousePosition(
        position="bottomleft",
        separator=" | ",
        prefix="Position:",
        num_digits=5,
        lat_formatter="function(lat) {return 'Lat: ' + lat.toFixed(5);}",
        lng_formatter="function(lng) {return 'Lon: ' + lng.toFixed(5);}"
    ).add_to(m)

    # Measurement tool
    MeasureControl(
        position="topleft",
        primary_length_unit="meters",
        secondary_length_unit="kilometers",
        primary_area_unit="sqmeters"
    ).add_to(m)

    # Mini overview map
    MiniMap(
        tile_layer="OpenStreetMap",
        position="bottomright",
        width=150, height=150,
        zoom_level_offset=-5
    ).add_to(m)

    # ── CUSTOM HTML PANEL (legend + stats) ──────────────────────
    n_pixels    = len(annual)
    mean_no2    = annual["NO2_pred_ugm3"].mean()
    pct_who     = (annual["NO2_pred_ugm3"] > WHO_LIMIT).mean() * 100
    pct_naaqs   = (annual["NO2_pred_ugm3"] > NAAQS_LIMIT).mean() * 100

    info_panel = f"""
    <div style='position:fixed;top:10px;right:10px;z-index:9999;
                background:white;border:1px solid #ccc;border-radius:8px;
                padding:12px 16px;font-family:Arial,sans-serif;
                font-size:12px;max-width:220px;
                box-shadow:2px 2px 8px rgba(0,0,0,0.15)'>

      <b style='font-size:14px'>Pune NO₂ Monitor</b><br>
      <span style='color:#666'>Sentinel-5P Downscaling</span>
      <hr style='margin:6px 0;border:none;border-top:1px solid #eee'>

      <b>Coverage</b><br>
      Pixels mapped: <b>{n_pixels:,}</b><br>
      Resolution: <b>500m</b><br>

      <hr style='margin:6px 0;border:none;border-top:1px solid #eee'>
      <b>Pune Annual Statistics</b><br>
      Mean NO₂: <b>{mean_no2:.1f} µg/m³</b><br>
      Above WHO (25): <b style='color:#ff8800'>{pct_who:.1f}%</b><br>
      Above NAAQS (80): <b style='color:#cc0000'>{pct_naaqs:.1f}%</b>

      <hr style='margin:6px 0;border:none;border-top:1px solid #eee'>
      <b>AQI Categories</b><br>
      {''.join([f"<span style='display:inline-block;width:12px;height:12px;background:{c};border:1px solid #999;margin-right:4px;vertical-align:middle'></span>{l}<br>"
                for _,_,c,l,_ in AQI_LEVELS])}

      <hr style='margin:6px 0;border:none;border-top:1px solid #eee'>
      <span style='color:#888;font-size:10px'>
        WHO guideline: 25 µg/m³<br>
        India NAAQS: 80 µg/m³<br>
        Click any pixel for details
      </span>
    </div>
    """

    m.get_root().html.add_child(folium.Element(info_panel))

    # ── LAYER CONTROL ────────────────────────────────────────────
    folium.LayerControl(
        position="topleft",
        collapsed=False
    ).add_to(m)

    return m


# =============================================================
#  SAVE HTML + GENERATE SUMMARY
# =============================================================

def save_map(m, df, output_path):
    m.save(output_path)
    file_size = os.path.getsize(output_path) / 1024**2

    annual = df.groupby(["latitude","longitude"],
                         as_index=False)["NO2_pred_ugm3"].mean()

    print(f"\n{'='*60}")
    print(f"  ✅ INTERACTIVE MAP SAVED")
    print(f"{'='*60}")
    print(f"  File:         {output_path}")
    print(f"  Size:         {file_size:.1f} MB")
    print(f"  Pixels:       {len(annual):,}")
    print(f"  Mean NO₂:     {annual['NO2_pred_ugm3'].mean():.1f} µg/m³")
    print(f"\n  HOW TO USE:")
    print(f"  1. Open {output_path} in Chrome/Firefox/Edge")
    print(f"  2. Use the layer switcher (top-left) to toggle views")
    print(f"  3. Click any coloured circle for full NO₂ details")
    print(f"  4. Hover over circles for quick values")
    print(f"  5. Use ruler tool (top-left) to measure distances")
    print(f"  6. Switch between Street/Satellite/Google tiles")
    print(f"  7. Mini-map (bottom-right) shows overview position")
    print(f"{'='*60}")


# =============================================================
#  MAIN
# =============================================================

if __name__ == "__main__":
    print("█"*60)
    print("  PUNE NO₂ — INTERACTIVE HEATMAP GENERATOR")
    print("█"*60)

    # Load predictions
    df = load_predictions()

    # Validate required columns
    required = ["latitude","longitude","NO2_pred_ugm3"]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    # Remove any invalid rows
    df = df[df["NO2_pred_ugm3"].between(0, 500)]
    df = df.dropna(subset=["latitude","longitude","NO2_pred_ugm3"])
    print(f"Clean rows for mapping: {len(df):,}")

    # Build and save map
    m = build_interactive_map(df)
    save_map(m, df, OUTPUT_HTML)

C:\Users\Admin\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\Admin\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


████████████████████████████████████████████████████████████
  PUNE NO₂ — INTERACTIVE HEATMAP GENERATOR
████████████████████████████████████████████████████████████
Loading predictions from: data/output/all_predictions_500m.csv
  Rows: 2,536,270  |  Columns: ['date', 'latitude', 'longitude', 'NO2_trop_umol_m2', 'NO2_pred_ugm3', 'season', 'month']
Clean rows for mapping: 2,536,270

Building interactive map...
  Unique pixels: 44,491
  NO2 range: 16.1 – 174.1 µg/m³
  Adding annual mean layer...
  Adding heatmap overlay layer...
  Adding seasonal layers...
  Adding NAAQS exceedance layer...
  Adding ground station markers...


MemoryError: 